# Suite TS5CONF — the TypeScript corpus

The conformance files this implementation writes for TypeScript 5 (`python -m mbse.Programs.Conformance.write`), as
for Python (P3CONF): each source's frame, the `References.OfFrame` of the transaction its parse gives, and the text its
program prints, for `typescript59.ts` and its JSX counterpart `typescript59-jsx.tsx`. The whole tree is values: the
frame binds only the workspace (`ws`) and the program (`program`), and its one assignment gives the program its `body`,
every statement nested in it, statements within terms among them, as one value. That the files are current, P3CONF-01
checks, the writer writing every language ported so far.

In [ ]:
import json

from mbse.Programs import TypeScript5
from mbse.Programs.Conformance import write
from mbse.Schemas.Framework import Comparison, Plain

CORPUS = write.DEFAULT
SOURCES = write.ROOT / "sources"
NAMES = ("typescript59.ts", "typescript59-jsx.tsx")

<a id="TS5CONF-01"></a>
## TS5CONF-01 · A frame read back compares equal, and is written again as it was

In [ ]:
for name in NAMES:
    stem = name.rsplit(".", 1)[0]
    text = (CORPUS / f"{stem}.json").read_text(encoding="utf-8")
    store = TypeScript5.store()
    schema = store.registered("References.OfFrame")
    mine = json.loads(write.frame(TypeScript5.store(), TypeScript5.Parse.parse((SOURCES / name).read_text(encoding="utf-8"), name)))
    read = Plain.FromPlain(store).Value(schema, json.loads(text))
    again = Plain.FromPlain(store).Value(schema, mine)
    assert Comparison.OfObject(schema, read).compare(Comparison.OfObject(schema, again)) == 0, name
    assert json.dumps(Plain.ToPlain(store).Value(schema, read), indent=2, ensure_ascii=False) + "\n" == text, name
    frame = json.loads(text)
    assert set(frame) == {"binds", "assigns", "scope", "journal"} and frame["scope"] == "0" and set(frame["binds"]) == {"ws", "program"}, name
    (assign,) = frame["assigns"]  # the program, given its whole tree as one nested value
    assert assign["object"] == frame["binds"]["program"] and set(assign["properties"]) <= {"hashbang", "body"}, name

<a id="TS5CONF-02"></a>
## TS5CONF-02 · The printed corpus parses to the tree its source does, and prints as it is

No TypeScript compiler is at hand to compare meanings, as `ast` does for Python, so the printed text is parsed again: its
tree is the source's but for the lines its statements begin on, and it prints back as it is.

In [ ]:
def unlined(value):
    """A plain value without its `line`s: where each statement begins, which the printer's layout moves."""
    if isinstance(value, dict):
        return {key: unlined(item) for key, item in value.items() if key != "line"}
    return [unlined(item) for item in value] if isinstance(value, list) else value


for name in NAMES:
    source = (SOURCES / name).read_text(encoding="utf-8")
    printed = (CORPUS / name).read_text(encoding="utf-8")
    trees = [json.loads(write.frame(TypeScript5.store(), TypeScript5.Parse.parse(text, name)))["assigns"][0]["properties"]
             for text in (source, printed)]
    assert unlined(trees[0]) == unlined(trees[1]), name
    store = TypeScript5.store()
    store.transaction(TypeScript5.Parse.parse(printed, name))
    assert TypeScript5.Print.print_module(store, name) == printed, name